# Experiment 3 — Explicit Task Instructions (Qwen)

**Model:** `qwen/qwen3.8-27b` via Groq API  
**Benchmark:** Same fixed 250-question obfuscated benchmark used in Experiments 1 and 2.

## Fixed benchmark
- FOL / obfuscation: 50
- Blood Relation / obfuscation_l1: 50
- Blood Relation / obfuscation_l2: 50
- Direction Sense / obfuscation: 50
- Number Series / obfuscation: 50
- **Total: 250**

## Method
**Explicit Task Instructions:** tell the model what the task is, what information to extract, and what task-specific structure/pattern to inspect before solving. The model must return only the requested final answer.

## Safety-first protocol
Run `TEST_MODE = True` first: 5 questions, one per variant. After checking the pipeline, change it to `False` for the full 250-question run.


In [ ]:
# 1. Install required packages
!pip -q install pandas openai tqdm


In [ ]:
# 2. Imports and configuration
import os
import re
import time
import random
import pandas as pd

from tqdm.auto import tqdm
from openai import OpenAI
from IPython.display import display

TEST_MODE = True
TEST_SAMPLES_PER_VARIANT = 1

MODEL_NAME = "qwen/qwen3.8-27b"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

TEMPERATURE = 0
MAX_TOKENS = 500

CHECKPOINT_EVERY_SUCCESSFUL = 10
REQUEST_DELAY_SECONDS = 1.5
MAX_RETRIES = 5

print("Model:", MODEL_NAME)
print("TEST_MODE:", TEST_MODE)


In [ ]:
# 3. Load Groq API key securely
GROQ_API_KEY = None

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = None

if not GROQ_API_KEY:
    from getpass import getpass
    GROQ_API_KEY = getpass("Enter your GROQ_API_KEY: ")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is required.")

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url=GROQ_BASE_URL,
)

print("Groq client ready.")


## 4. Upload the SAME benchmark used in Experiments 1 and 2

Upload:

`qwen_obfuscated_benchmark_50_per_variant_250.csv`

Do not make a new sample. The benchmark must remain fixed across Experiments 1, 2, and 3.


In [ ]:
# 4. Upload benchmark CSV
from google.colab import files

uploaded = files.upload()

if not uploaded:
    raise ValueError("No file was uploaded.")

csv_names = [
    name for name in uploaded.keys()
    if name.lower().endswith(".csv")
]

if not csv_names:
    raise ValueError("Please upload the 250-question benchmark CSV.")

csv_name = csv_names[0]
df = pd.read_csv(csv_name)

print("Loaded:", csv_name)
print("Rows:", len(df))
print("Columns:", list(df.columns))

display(df.head())


In [ ]:
# 5. Validate the fixed 250-question benchmark
required_columns = [
    "dataset",
    "variant",
    "prompt",
    "ground_truth",
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

expected = {
    ("FOL", "obfuscation"): 50,
    ("Blood_Relation", "obfuscation_l1"): 50,
    ("Blood_Relation", "obfuscation_l2"): 50,
    ("Direction_Sense", "obfuscation"): 50,
    ("Number_Series", "obfuscation"): 50,
}

distribution = (
    df.groupby(["dataset", "variant"])
      .size()
      .reset_index(name="count")
)

print("Benchmark distribution:")
display(distribution)

for (dataset, variant), expected_count in expected.items():
    actual_count = int(
        (
            (df["dataset"].astype(str) == dataset) &
            (df["variant"].astype(str) == variant)
        ).sum()
    )
    print(
        f"{dataset} / {variant}: "
        f"{actual_count} / {expected_count}"
    )
    if actual_count != expected_count:
        raise ValueError(
            f"Unexpected count for {dataset}/{variant}: "
            f"{actual_count}; expected {expected_count}."
        )

if len(df) != 250:
    raise ValueError(f"Expected 250 total rows, found {len(df)}.")

if "response_id" in df.columns:
    duplicate_ids = df["response_id"].astype(str).duplicated().sum()
    print("Duplicate response_id values:", duplicate_ids)
    if duplicate_ids != 0:
        raise ValueError("response_id must be unique in the benchmark.")

print("\n✅ Fixed 250-question benchmark validated.")


In [ ]:
# 6. Explicit Task Instructions prompts

FOL_SYSTEM = r'''
You are solving an obfuscated first-order-logic reasoning problem.

EXPLICIT TASK INSTRUCTIONS:
1. Identify the entities, predicates, facts, rules, quantifiers, and constraints.
2. Translate the obfuscated wording into the clearest logical relations you can infer
   without adding facts.
3. Determine exactly what the question asks you to establish or select.
4. Use the extracted constraints to solve the problem.
5. Check the conclusion against the relevant facts and constraints.

Return ONLY the final answer requested by the question.
Do not output your reasoning, translation, explanation, labels, or extra text.
'''

BLOOD_RELATION_SYSTEM = r'''
You are solving an obfuscated blood-relation reasoning problem.

EXPLICIT TASK INSTRUCTIONS:
1. Identify every person/entity mentioned.
2. Identify each stated family relationship.
3. Pay attention to the direction of each relationship.
4. Build the relevant family chain/graph.
5. Determine exactly which relationship the question asks for.
6. Check the final relationship against the complete chain.

Return ONLY the final answer requested by the question.
Do not output your reasoning, relationship chain, explanation, labels, or extra text.
'''

DIRECTION_SYSTEM = r'''
You are solving an obfuscated direction-sense reasoning problem.

EXPLICIT TASK INSTRUCTIONS:
1. Identify the starting/reference position.
2. Extract every movement and its direction in the order given.
3. Convert the movements into relative horizontal/vertical displacement.
4. Combine the movements carefully.
5. Determine exactly what the question asks: final direction, relative position,
   distance, or starting-point relation.
6. Check units and the final displacement before answering.

Return ONLY the final answer requested by the question.
Do not output your calculations, explanation, labels, or extra text.
'''

NUMBER_SERIES_SYSTEM = r'''
You are solving an obfuscated number-series reasoning problem.

EXPLICIT TASK INSTRUCTIONS:
1. Read the entire sequence before answering.
2. Inspect differences, ratios, alternating operations, positional relations,
   and other plausible numerical transformations.
3. Identify the simplest pattern that explains the available terms.
4. Verify the pattern across multiple transitions, not just one pair.
5. Determine exactly which term/value the question asks for.
6. Apply the verified pattern and return the requested value.

Return ONLY the final answer requested by the question.
Do not output your reasoning, pattern explanation, labels, or extra text.
'''

SYSTEM_PROMPTS = {
    "FOL": FOL_SYSTEM,
    "Blood_Relation": BLOOD_RELATION_SYSTEM,
    "Direction_Sense": DIRECTION_SYSTEM,
    "Number_Series": NUMBER_SERIES_SYSTEM,
}

print("Explicit task-instruction prompts ready.")


In [ ]:
# 7. Answer normalization and model-call helpers

def normalize_answer(value):
    if value is None or pd.isna(value):
        return ""

    text = str(value).strip()

    prefixes = [
        "Final Answer:",
        "Final answer:",
        "Answer:",
        "answer:",
        "The answer is:",
        "the answer is:",
    ]

    for prefix in prefixes:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()
            break

    text = re.sub(r"\s+", " ", text).strip().lower()
    text = text.strip(" \t\n\r\"'.,;:!?")

    return text


def run_model(prompt, dataset):
    system_prompt = SYSTEM_PROMPTS.get(str(dataset))

    if not system_prompt:
        raise ValueError(
            f"No explicit-task prompt configured for dataset: {dataset}"
        )

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": str(prompt)},
        ],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )

    content = response.choices[0].message.content

    if content is None:
        return ""

    return str(content).strip()


def is_rate_limit_or_transient_error(exc):
    text = repr(exc).lower()
    keywords = [
        "rate limit", "429", "too many requests",
        "timeout", "temporarily",
        "service unavailable", "502", "503", "504",
    ]
    return any(k in text for k in keywords)


print("Model helpers ready.")


In [ ]:
# 8. Select the 5-question safety test or the full 250-question benchmark

variant_order = [
    ("FOL", "obfuscation"),
    ("Blood_Relation", "obfuscation_l1"),
    ("Blood_Relation", "obfuscation_l2"),
    ("Direction_Sense", "obfuscation"),
    ("Number_Series", "obfuscation"),
]

if TEST_MODE:
    parts = []

    for dataset_name, variant_name in variant_order:
        subset = df[
            (df["dataset"].astype(str) == dataset_name) &
            (df["variant"].astype(str) == variant_name)
        ]
        if len(subset) < TEST_SAMPLES_PER_VARIANT:
            raise ValueError(
                f"Not enough rows for {dataset_name}/{variant_name}."
            )
        parts.append(subset.head(TEST_SAMPLES_PER_VARIANT))

    run_df = pd.concat(parts, ignore_index=True)
    print("TEST MODE")
    print("Rows to process:", len(run_df))
else:
    run_df = df.copy().reset_index(drop=True)
    print("FULL MODE")
    print("Rows to process:", len(run_df))
    if len(run_df) != 250:
        raise ValueError(f"Expected 250 rows, found {len(run_df)}.")

display(
    run_df[
        ["dataset", "variant", "row_number", "response_id", "ground_truth"]
    ]
)


In [ ]:
# 9. Run inference — failed rows remain repairable

results = []

checkpoint_file = (
    None if TEST_MODE
    else "experiment_03_explicit_task_instructions_Qwen_250_checkpoint.csv"
)

if not TEST_MODE and checkpoint_file and os.path.exists(checkpoint_file):
    previous = pd.read_csv(checkpoint_file)
    previous["prediction"] = (
        previous["prediction"].fillna("").astype(str).str.strip()
    )

    completed_previous = previous[
        previous["prediction"] != ""
    ].copy()

    results = completed_previous.to_dict("records")
    processed_ids = set(
        completed_previous["response_id"].astype(str).tolist()
    )

    print(
        f"Resuming from checkpoint: "
        f"{len(completed_previous)} completed predictions."
    )
else:
    processed_ids = set()
    if not TEST_MODE:
        print("No usable checkpoint found. Starting from row 1.")

print("\nStarting inference...")

for _, row in tqdm(
    run_df.iterrows(),
    total=len(run_df),
    desc="Qwen Explicit Task Instructions"
):

    response_id = str(row.get("response_id", ""))

    if not TEST_MODE and response_id in processed_ids:
        continue

    prediction = ""
    error = ""

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            prediction = run_model(
                row["prompt"],
                row["dataset"],
            )

            if prediction.strip():
                error = ""
                break

            error = "EMPTY_MODEL_RESPONSE"

        except Exception as e:
            error = repr(e)

            if attempt < MAX_RETRIES:
                wait_seconds = min(
                    60,
                    5 * (2 ** (attempt - 1))
                ) + random.uniform(0, 2)

                if is_rate_limit_or_transient_error(e):
                    print(
                        f"\nTransient/rate-limit error for "
                        f"{response_id}, attempt "
                        f"{attempt}/{MAX_RETRIES}."
                    )
                    print(
                        f"Waiting {wait_seconds:.1f}s before retry..."
                    )
                    time.sleep(wait_seconds)
                else:
                    print(
                        f"\nAPI error for {response_id}, "
                        f"attempt {attempt}/{MAX_RETRIES}: {e}"
                    )
                    time.sleep(2)
            else:
                print(
                    f"\nFailed after {MAX_RETRIES} attempts: "
                    f"{response_id}"
                )

    ground_truth = normalize_answer(row["ground_truth"])
    prediction_clean = normalize_answer(prediction)

    correct = (
        bool(ground_truth)
        and bool(prediction_clean)
        and prediction_clean == ground_truth
    )

    results.append({
        "dataset": row["dataset"],
        "variant": row["variant"],
        "setting": row.get("setting", ""),
        "row_number": row.get("row_number", ""),
        "response_id": response_id,
        "ground_truth": ground_truth,
        "prediction": prediction_clean,
        "correct": correct,
        "error": error if not prediction_clean else "",
    })

    if not TEST_MODE:
        temp_df = pd.DataFrame(results)

        successful_count = int(
            temp_df["prediction"]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
            .sum()
        )

        if (
            successful_count > 0
            and successful_count % CHECKPOINT_EVERY_SUCCESSFUL == 0
        ):
            temp_df.to_csv(
                checkpoint_file,
                index=False
            )
            print(
                f"\nCheckpoint saved: "
                f"{successful_count}/250 successful predictions."
            )

    time.sleep(REQUEST_DELAY_SECONDS)

results_df = pd.DataFrame(results)

print("\n" + "=" * 60)
print("INFERENCE PASS COMPLETE")
print("=" * 60)
print("Rows in results:", len(results_df))
print(
    "Predictions:",
    results_df["prediction"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)
print(
    "Empty predictions:",
    results_df["prediction"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)
print(
    "Errors:",
    results_df["error"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)


In [ ]:
# 10. Inspect the inference results

display(
    results_df[
        [
            "dataset",
            "variant",
            "setting",
            "row_number",
            "ground_truth",
            "prediction",
            "correct",
            "error",
        ]
    ]
)


In [ ]:
# 11. Final completeness / error check

results_df["prediction"] = (
    results_df["prediction"]
    .fillna("")
    .astype(str)
    .str.strip()
)

results_df["error"] = (
    results_df["error"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Clear stale API errors once a prediction exists.
results_df.loc[
    results_df["prediction"] != "",
    "error"
] = ""

missing_count = int(
    results_df["prediction"].eq("").sum()
)

error_count = int(
    results_df["error"].ne("").sum()
)

print("=" * 60)
print("COMPLETENESS CHECK")
print("=" * 60)
print("Total rows:", len(results_df))
print("Predictions:", int(results_df["prediction"].ne("").sum()))
print("Missing predictions:", missing_count)
print("Unresolved errors:", error_count)

if not TEST_MODE:
    if len(results_df) != 250:
        print("⚠️ Results do not yet contain all 250 rows.")

    if missing_count == 0 and error_count == 0:
        print("\n✅ All 250 predictions are complete and error-free.")
    else:
        print(
            "\n⚠️ Do NOT treat this as final yet. "
            "Repair missing/error rows first."
        )


In [ ]:
# 12. Accuracy by Task + Variant

valid = results_df[
    results_df["ground_truth"].astype(str).str.strip().ne("") &
    results_df["prediction"].astype(str).str.strip().ne("")
].copy()

if len(valid) == 0:
    print("No valid prediction/ground-truth pairs available.")
else:
    overall_accuracy = valid["correct"].mean() * 100
    print(f"Overall accuracy: {overall_accuracy:.2f}%")

    variant_summary = (
        valid
        .groupby(["dataset", "variant"])["correct"]
        .agg(
            total="count",
            correct="sum",
            accuracy="mean"
        )
        .reset_index()
    )

    variant_summary["accuracy_percent"] = (
        variant_summary["accuracy"] * 100
    )

    variant_summary = variant_summary[
        [
            "dataset",
            "variant",
            "total",
            "correct",
            "accuracy_percent",
        ]
    ]

    print("\nTask + Variant results:")
    display(variant_summary)


In [ ]:
# 13. Save results

if TEST_MODE:
    results_file = (
        "experiment_03_explicit_task_instructions_Qwen_TEST_results.csv"
    )
    summary_file = (
        "experiment_03_explicit_task_instructions_Qwen_TEST_summary.csv"
    )
else:
    results_file = (
        "experiment_03_explicit_task_instructions_Qwen_250_results.csv"
    )
    summary_file = (
        "experiment_03_explicit_task_instructions_Qwen_250_summary.csv"
    )

results_df.to_csv(results_file, index=False)

if len(valid) > 0:
    variant_summary.to_csv(summary_file, index=False)

print("Saved:", results_file)

if len(valid) > 0:
    print("Saved:", summary_file)

if checkpoint_file:
    print("Checkpoint:", checkpoint_file)


In [ ]:
# 14. Download results to your laptop

from google.colab import files

files.download(results_file)

if len(valid) > 0:
    time.sleep(2)
    files.download(summary_file)

print("✅ Download started.")


## Run instructions

### Safety test
Keep:

```python
TEST_MODE = True
```

Run through the inference and completeness cells. The test processes exactly 5 questions:
- FOL / obfuscation
- Blood Relation / obfuscation_l1
- Blood Relation / obfuscation_l2
- Direction Sense / obfuscation
- Number Series / obfuscation

Confirm that predictions are generated and the pipeline is healthy.

### Full run
Then change:

```python
TEST_MODE = False
```

Run the configuration/validation/selection and inference cells.

The full run uses the same 250-question benchmark as Experiments 1 and 2. It checkpoints successful predictions only, so failed rows remain repairable. If rate limits occur, do not rerun all 250; repair only missing rows after the API limit recovers.

Final results are reported by **Task + Variant**.
